In [1]:
import torch, transformers
print(torch.__version__, torch.cuda.is_available())
print(transformers.__version__)

c:\Users\Home\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.13.0+cu130 True
5.15.1


In [2]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    'D:/models/olmoe-hf',
    dtype=torch.float16,
    device_map='cpu'
)

Loading weights: 100%|██████████| 179/179 [05:45<00:00,  1.93s/it]


In [4]:
gate = model.model.layers[0].mlp.gate
print(gate)

print("\nParamètres :")
for name, param in gate.named_parameters():
    print(f"  {name}: shape={tuple(param.shape)}")

OlmoeTopKRouter()

Paramètres :
  weight: shape=(64, 2048)


In [5]:
import inspect
print(inspect.getsource(gate.forward))

    def forward(self, hidden_states):
        hidden_states = hidden_states.reshape(-1, self.hidden_dim)
        router_logits = F.linear(hidden_states, self.weight)  # (seq_len, num_experts)
        router_probs = torch.nn.functional.softmax(router_logits, dtype=torch.float, dim=-1)
        router_top_value, router_indices = torch.topk(router_probs, self.top_k, dim=-1)  # (seq_len, top_k)
        if self.norm_topk_prob:
            router_top_value /= router_top_value.sum(dim=-1, keepdim=True)
        router_top_value = router_top_value.to(router_logits.dtype)
        router_scores = router_top_value
        return router_logits, router_scores, router_indices



In [6]:
def compute_hobbit_importance(router_scores):
    """
    router_scores: tensor de shape (top_k,) — les probabilités normalisées
                   d'UN SEUL token, déjà triées décroissant par torch.topk.

    Retourne : unimportance_scores, où une valeur PROCHE DE 0 = expert important
               (à garder en haute précision), une valeur PROCHE DE 1 = peu important
               (candidat à la basse précision en cas de cache miss).
    """
    # Équation 2 de HOBBIT : score cumulé des experts classés AVANT celui-ci
    cumulative = torch.cumsum(router_scores, dim=0)
    unimportance = cumulative - router_scores  # exclut le score de l'expert lui-même
    unimportance[0] = 0.0  # le tout premier expert (e0) est toujours jugé important
    return unimportance

# Test avec notre exemple à la main (E3=0.72, E0=0.16, E6=0.12)
exemple_scores = torch.tensor([0.72, 0.16, 0.12])
print(compute_hobbit_importance(exemple_scores))
# Attendu : tensor([0.00, 0.72, 0.88])

tensor([0.0000, 0.7200, 0.8800])


In [7]:
# Un texte d'exemple qu'on va faire passer dans le tokenizer + la première couche
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('D:/models/olmoe-hf')
inputs = tokenizer("Le chat mange une souris", return_tensors="pt")

# On récupère juste l'embedding d'entrée (avant la couche 0) pour simplifier ce premier test
with torch.no_grad():
    hidden = model.model.embed_tokens(inputs['input_ids'])
    router_logits, router_scores, router_indices = model.model.layers[0].mlp.gate(hidden)

print("Scores du premier token :", router_scores[0])
print("Experts sélectionnés :", router_indices[0])
print("Scores d'importance HOBBIT :", compute_hobbit_importance(router_scores[0]))

Scores du premier token : tensor([0.0165, 0.0163, 0.0162, 0.0161, 0.0161, 0.0161, 0.0159, 0.0159],
       dtype=torch.float16)
Experts sélectionnés : tensor([57, 61, 46, 19,  3, 44,  4, 22])
Scores d'importance HOBBIT : tensor([0.0000, 0.0165, 0.0328, 0.0490, 0.0651, 0.0811, 0.0972, 0.1130],
       dtype=torch.float16)


In [8]:
captured = {}

def hook_fn(module, input, output):
    router_logits, router_scores, router_indices = output
    captured['scores'] = router_scores
    captured['indices'] = router_indices

# On attache le hook au routeur de la couche 0
handle = model.model.layers[0].mlp.gate.register_forward_hook(hook_fn)

# Un vrai passage complet dans le modèle (embedding → attention → norm → routeur → ...)
with torch.no_grad():
    model(**inputs)

handle.remove()  # on retire le hook, propreté

print("Scores réels (après le vrai pipeline complet) :", captured['scores'][0])
print("Experts sélectionnés :", captured['indices'][0])
print("Scores d'importance HOBBIT :", compute_hobbit_importance(captured['scores'][0]))

Scores réels (après le vrai pipeline complet) : tensor([0.1345, 0.1063, 0.0921, 0.0855, 0.0829, 0.0705, 0.0682, 0.0214],
       dtype=torch.float16)
Experts sélectionnés : tensor([46, 11, 45, 17, 57, 47,  3, 34])
Scores d'importance HOBBIT : tensor([0.0000, 0.1345, 0.2410, 0.3330, 0.4185, 0.5015, 0.5718, 0.6396],
       dtype=torch.float16)
